In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [1]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report
from transformers import BertTokenizer, AutoModelForSequenceClassification, get_scheduler
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW
from tqdm.auto import tqdm
import torch
import os

In [ ]:
# Set HF token
os.environ["HF_TOKEN"] = ""

In [ ]:
# 1. Load dan preprocess data
df = pd.read_csv("")

In [4]:
# 2. Buat label encoding
label2id = {label: i for i, label in enumerate(df['emosi'].unique())}
id2label = {i: label for label, i in label2id.items()}
df['label'] = df['emosi'].map(label2id)


In [5]:
# 3. Hapus duplikat
df = df.drop_duplicates(subset='clean_content', keep='first')
print(f"Data setelah hapus duplikat: {len(df)}")
print("Distribusi kelas:")
print(df['emosi'].value_counts())

Data setelah hapus duplikat: 15751
Distribusi kelas:
emosi
Marah     5274
Kecewa    4662
Senang    3806
Takut     2009
Name: count, dtype: int64


In [6]:
# 4. Split data 80/10/10
texts = df['clean_content'].tolist()
labels = df['label'].tolist()

X_train, X_temp, y_train, y_temp = train_test_split(
    texts, labels, test_size=0.2, random_state=42, stratify=labels
)
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.5, random_state=42, stratify=y_temp
)

print(f"Train: {len(X_train)}, Val: {len(X_val)}, Test: {len(X_test)}")


Train: 12600, Val: 1575, Test: 1576


In [7]:
# 5. Initialize tokenizer
tokenizer = BertTokenizer.from_pretrained("indobenchmark/indobert-base-p1")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/2.00 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json: 0.00B [00:00, ?B/s]

In [36]:
# 6. Dataset class
class ReviewDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_length=128):
        self.texts = texts
        self.labels = labels
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        text = str(self.texts[idx])
        encoding = self.tokenizer(
            text,
            truncation=True,
            padding='max_length',
            max_length=self.max_length,
            return_tensors='pt'
        )

        return {
            'input_ids': encoding['input_ids'].flatten(),
            'attention_mask': encoding['attention_mask'].flatten(),
            'labels': torch.tensor(self.labels[idx], dtype=torch.long)
        }

In [37]:
# 7. Buat datasets dan dataloaders
train_dataset = ReviewDataset(X_train, y_train, tokenizer)
val_dataset = ReviewDataset(X_val, y_val, tokenizer)
test_dataset = ReviewDataset(X_test, y_test, tokenizer)

train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=16)
test_loader = DataLoader(test_dataset, batch_size=16)



In [56]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

model = AutoModelForSequenceClassification.from_pretrained(
    "indobenchmark/indobert-base-p1",
    num_labels=len(label2id)
).to(device)

# Perbaikan di sini
optimizer = AdamW(model.parameters(), lr=3e-5, weight_decay=0.01)
num_training_steps = 3 * len(train_loader)
num_warmup_steps = int(0.1 * num_training_steps)  # 10% warmup

scheduler = get_scheduler(
    "linear",
    optimizer=optimizer,
    num_warmup_steps=num_warmup_steps,
    num_training_steps=num_training_steps
)

Using device: cuda


Some weights of BertForSequenceClassification were not initialized from the model checkpoint at indobenchmark/indobert-base-p1 and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [57]:
# 9. Training loop
print("\nMulai training...")
model.train()

for epoch in range(2):
    total_loss = 0
    progress_bar = tqdm(train_loader, desc=f"Epoch {epoch+1}")

    for batch in progress_bar:
        batch = {k: v.to(device) for k, v in batch.items()}

        optimizer.zero_grad()
        outputs = model(**batch)
        loss = outputs.loss
        loss.backward()
        optimizer.step()
        scheduler.step()

        total_loss += loss.item()
        progress_bar.set_postfix(loss=loss.item())

    avg_loss = total_loss / len(train_loader)
    print(f"Epoch {epoch+1} - Average Loss: {avg_loss:.4f}")




Mulai training...


Epoch 1:   0%|          | 0/788 [00:00<?, ?it/s]

Epoch 1 - Average Loss: 0.6568


Epoch 2:   0%|          | 0/788 [00:00<?, ?it/s]

Epoch 2 - Average Loss: 0.3595


In [58]:
# 10. Evaluasi pada validation set
print("\nEvaluasi validation...")
model.eval()
val_predictions = []
val_true_labels = []
val_losses = []

for batch in tqdm(val_loader, desc="Validation"):
    batch = {k: v.to(device) for k, v in batch.items()}

    with torch.no_grad():
        outputs = model(**batch)
        loss = outputs.loss                # <- ambil loss
        predictions = torch.argmax(outputs.logits, dim=-1)

        val_losses.append(loss.item())
        val_predictions.extend(predictions.cpu().numpy())
        val_true_labels.extend(batch["labels"].cpu().numpy())

# Hitung rata-rata loss & akurasi
val_loss = sum(val_losses) / len(val_losses)
val_accuracy = accuracy_score(val_true_labels, val_predictions)

print(f"Validation Loss: {val_loss:.4f}")
print(f"Validation Accuracy: {val_accuracy:.4f}")



Evaluasi validation...


Validation:   0%|          | 0/99 [00:00<?, ?it/s]

Validation Loss: 0.4397
Validation Accuracy: 0.8095


In [59]:
# 11. Classification report
class_names = [id2label[i] for i in range(len(label2id))]
print("\nValidation Classification Report:")
print(classification_report(val_true_labels, val_predictions, target_names=class_names))


Validation Classification Report:
              precision    recall  f1-score   support

       Takut       0.80      0.90      0.85       201
      Senang       0.92      0.94      0.93       381
      Kecewa       0.77      0.69      0.72       466
       Marah       0.77      0.79      0.78       527

    accuracy                           0.81      1575
   macro avg       0.81      0.83      0.82      1575
weighted avg       0.81      0.81      0.81      1575



In [60]:
# 12. Evaluasi test set
print("\nEvaluasi test set...")
test_predictions = []
test_true_labels = []

for batch in tqdm(test_loader, desc="Testing"):
    batch = {k: v.to(device) for k, v in batch.items()}

    with torch.no_grad():
        outputs = model(**batch)
        predictions = torch.argmax(outputs.logits, dim=-1)

        test_predictions.extend(predictions.cpu().numpy())
        test_true_labels.extend(batch["labels"].cpu().numpy())

test_accuracy = accuracy_score(test_true_labels, test_predictions)
print(f"Test Accuracy: {test_accuracy:.4f}")

print("\nTest Classification Report:")
print(classification_report(test_true_labels, test_predictions, target_names=class_names))


Evaluasi test set...


Testing:   0%|          | 0/99 [00:00<?, ?it/s]

Test Accuracy: 0.8230

Test Classification Report:
              precision    recall  f1-score   support

       Takut       0.79      0.80      0.79       201
      Senang       0.95      0.96      0.95       380
      Kecewa       0.80      0.71      0.75       467
       Marah       0.77      0.84      0.80       528

    accuracy                           0.82      1576
   macro avg       0.83      0.83      0.82      1576
weighted avg       0.82      0.82      0.82      1576

